# Building an Ensemble Model

This tutorial walks through the complete ensemble workflow in `ncaa_eval`:

1. **Define** base models (XGBoost + Elo) and a meta-learner (Logistic Regression)
2. **Train** the stacked ensemble with a single `run_training()` call
3. **Compare** OOF (out-of-fold) log loss across base models and the ensemble
4. **Generate** a bracket probability matrix for a target season
5. **Export** a Kaggle-format submission CSV

### Prerequisites

- The `ncaa_eval` package is installed (`pip install -e .` from the repo root)
- Data has been synced via `python -m ncaa_eval.cli sync` (Kaggle data must be present in `data/`)
- The `ncaa_eval` conda environment is active

### Expected Training Time

The ensemble training step takes approximately 2–5 minutes. It runs 10 walk-forward
backtests per base model (2 models × 10 seasons), then trains the meta-learner and
retrains both base models on the full dataset.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import log_loss

from ncaa_eval.cli.train import run_training
from ncaa_eval.evaluation.kaggle_export import format_kaggle_submission
from ncaa_eval.model import StackedEnsemble
from ncaa_eval.model.elo import EloModel
from ncaa_eval.model.logistic_regression import LogisticRegressionModel
from ncaa_eval.model.tracking import RunStore
from ncaa_eval.model.xgboost_model import XGBoostModel

/home/dhilg/miniforge3/envs/ncaa_eval/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


## 1. Define Base Models

A stacked ensemble combines **complementary** base models. We pair:

- **XGBoost** — a gradient-boosted tree that excels at capturing non-linear
  relationships between per-game features (SRS ratings, seed differences, etc.)
- **Elo** — a stateful rating system that tracks team strength over time,
  capturing momentum and schedule-strength effects that per-game snapshots miss

Together they give the meta-learner two distinct "views" of each matchup.

In [2]:
xgb = XGBoostModel(batch_rating_types=("srs",))
elo = EloModel()

print(f"XGBoost config: {xgb.get_config().model_name}")
print(f"Elo config:     {elo.get_config().model_name}")

XGBoost config: xgboost
Elo config:     elo


## 2. Define the Meta-Learner

The meta-learner is a simple model that learns how to **blend** the base model
predictions. Logistic regression works well here — it learns a calibrated weighted
average of the base model probabilities plus a few contextual features.

In [3]:
meta = LogisticRegressionModel()

## 3. Construct the Stacked Ensemble

The `StackedEnsemble` ties the base models and meta-learner together. It also
accepts `contextual_features` — game-level features passed directly to the
meta-learner alongside the base model predictions.

The defaults are:
- `seed_diff` — tournament seed difference (higher seed − lower seed)
- `is_tournament` — whether the game is a tournament game
- `loc_encoding` — home/away/neutral encoding

These give the meta-learner context about *where* and *when* a game is played,
which can shift how much weight it gives to each base model.

In [4]:
ensemble = StackedEnsemble(
    base_models=[xgb, elo],
    meta_learner=meta,
    # contextual_features defaults to ["seed_diff", "is_tournament", "loc_encoding"]
)

print(f"Base models:          {[m.get_config().model_name for m in ensemble.base_models]}")
print(f"Meta-learner:         {ensemble.meta_learner.get_config().model_name}")
print(f"Contextual features:  {ensemble.contextual_features}")

Base models:          ['xgboost', 'elo']
Meta-learner:         logistic_regression
Contextual features:  ['seed_diff', 'is_tournament', 'loc_encoding']


## 4. Train the Ensemble

A single `run_training()` call handles the full 6-step ensemble training pipeline:

1. Generate out-of-fold (OOF) predictions for each base model via walk-forward CV
2. Align OOF predictions by `game_id` across base models
3. Build the meta-learner training set (base model OOF predictions + contextual features)
4. Train the meta-learner on the aligned OOF data
5. Retrain each base model on the full training set
6. Persist everything to disk

The cell below trains on seasons 2015–2024. This takes roughly 2–5 minutes.

In [5]:
data_dir = Path("../../data")
output_dir = Path("../../output")

run = run_training(
    ensemble,
    data_dir=data_dir,
    start_year=2015,
    end_year=2024,
    output_dir=output_dir,
    model_name="tutorial_ensemble",
)

print(f"\nRun ID:     {run.run_id}")
print(f"Model type: {run.model_type}")

Step 1/6: Generating OOF predictions...

OOF backtest for base model 0 (XGBoostModel)...

Running backtest: 8 folds, n_jobs=1

                       Backtest Results                        
┏━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━┓
┃ Year ┃ brier_score ┃ ece    ┃ log_loss ┃ roc_auc ┃ Time (s) ┃
┡━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━┩
│ 2016 │ 0.1867      │ 0.0740 │ 0.5586   │ 0.7926  │ 4.90     │
│ 2017 │ 0.1694      │ 0.1095 │ 0.4957   │ 0.8185  │ 0.53     │
│ 2018 │ 0.1945      │ 0.1321 │ 0.5688   │ 0.7858  │ 1.06     │
│ 2019 │ 0.1558      │ 0.1273 │ 0.4653   │ 0.8666  │ 0.64     │
│ 2021 │ 0.2244      │ 0.1818 │ 0.6547   │ 0.7397  │ 1.27     │
│ 2022 │ 0.2224      │ 0.1416 │ 0.6604   │ 0.7132  │ 1.67     │
│ 2023 │ 0.2145      │ 0.0994 │ 0.6580   │ 0.7214  │ 1.43     │
│ 2024 │ 0.1901      │ 0.1257 │ 0.5609   │ 0.7976  │ 0.90     │
└──────┴─────────────┴────────┴──────────┴─────────┴──────────┘

Total backtest time: 15.53s

...

Retraining base model 0 on full dataset...

Retraining base model 1 on full dataset...

Step 6/6: Persisting ensemble artifacts...

               Ensemble Training Results                
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Field         ┃ Value                                ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Run ID        │ 94441188-7f63-4ddc-a054-2ccb52204aae │
│ Model         │ tutorial_ensemble                    │
│ Base models   │ 2                                    │
│ Seasons       │ 2015–2024                            │
│ OOF games     │ 535                                  │
│ Meta features │ 5                                    │
│ Git hash      │ 0e36a2f                              │
└───────────────┴──────────────────────────────────────┘


Run ID:     94441188-7f63-4ddc-a054-2ccb52204aae
Model type: ensemble


## 5. Compare OOF Performance

During training, each base model produced out-of-fold predictions via
walk-forward cross-validation. The `oof_aligned.parquet` file contains:

- **`pred_base_0`, `pred_base_1`** — truly OOF predictions: each base model
  was trained on folds that excluded the prediction season.
- **`pred_ensemble`** — the meta-learner's in-sample predictions on those
  OOF base-model predictions. Because logistic regression has low variance,
  in-sample ≈ out-of-sample, but this is not a nested CV estimate.

If the ensemble is working well, its log loss should **match or beat** every
individual base model.

In [6]:
store = RunStore(output_dir)

# Load the ensemble manifest and OOF aligned data
model_path = store.model_dir(run.run_id)
manifest = json.loads((model_path / "manifest.json").read_text())
oof = pd.read_parquet(model_path / "oof_aligned.parquet")

print("Manifest contents:")
for key, value in manifest.items():
    print(f"  {key}: {value}")
print(f"\nOOF aligned games: {len(oof)}")

Manifest contents:
  base_model_types: ['xgboost', 'elo']
  base_model_count: 2
  contextual_features: ['seed_diff', 'is_tournament', 'loc_encoding']
  meta_learner_type: logistic_regression
  meta_column_order: ['pred_base_0', 'pred_base_1', 'seed_diff', 'is_tournament', 'loc_encoding']
  oof_backtest_run_ids: ['f3148b72-3305-4da1-b936-51eae69efa41', '00f20a17-293b-4b84-87a9-97f54a1eb956']
  oof_game_count: 535
  oof_drop_pct: 0.0

OOF aligned games: 535


In [7]:
# Use sklearn's log_loss (same metric as the dashboard leaderboard)
y = oof["team_a_won"].values

rows = []
for i, model_type in enumerate(manifest["base_model_types"]):
    pred_col = f"pred_base_{i}"
    rows.append({
        "model": model_type,
        "oof_log_loss": round(log_loss(y, oof[pred_col].values), 5),
    })

rows.append({
    "model": "ensemble",
    "oof_log_loss": round(log_loss(y, oof["pred_ensemble"].values), 5),
})

comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))

   model  oof_log_loss
 xgboost       0.57766
     elo       0.61426
ensemble       0.55478


The ensemble should show a lower (better) log loss than either base model alone.
This is the core benefit of stacking — the meta-learner learns to combine the
strengths of each base model while mitigating their individual weaknesses.

## 6. Generate Bracket Predictions

The `predict_bracket()` method generates an n×n probability matrix for all
tournament-eligible teams in a given season. Each cell `P[a, b]` is the
estimated probability that team `a` beats team `b`.

The matrix is symmetric: `P[a, b] + P[b, a] ≈ 1.0`, and the diagonal is `0`.

In [8]:
prob_matrix = ensemble.predict_bracket(data_dir, season=2025)

print(f"Matrix shape: {prob_matrix.shape}")
print(f"Teams: {len(prob_matrix)} tournament-eligible teams")
print()
print("Sample (first 5 teams, 5 columns):")
print(prob_matrix.iloc[:5, :5].to_string(float_format="{:.3f}".format))

Matrix shape: (364, 364)
Teams: 364 tournament-eligible teams

Sample (first 5 teams, 5 columns):
      1101  1102  1103  1104  1105
1101 0.000 0.703 0.170 0.170 0.714
1102 0.297 0.000 0.112 0.095 0.438
1103 0.830 0.888 0.000 0.367 0.817
1104 0.830 0.905 0.633 0.000 0.828
1105 0.286 0.562 0.183 0.172 0.000


## 7. Export a Kaggle Submission

The Kaggle March Machine Learning Mania competition expects a CSV with columns
`ID` and `Pred`, where `ID` has the format `YYYY_TeamID1_TeamID2` (lower ID first)
and `Pred` is the probability that the lower-ID team wins.

`format_kaggle_submission()` handles the formatting automatically.

In [9]:
team_ids = list(prob_matrix.index)
csv_str = format_kaggle_submission(2025, team_ids, prob_matrix.to_numpy())

# Write to file
submission_path = Path("../../output/tutorial_ensemble_submission.csv")
submission_path.write_text(csv_str)
print(f"Submission written to: {submission_path.resolve()}")

Submission written to: /home/dhilg/git/NCAA_eval/output/tutorial_ensemble_submission.csv


In [10]:
# Preview the first few rows
lines = csv_str.strip().split("\n")
print(f"Total rows: {len(lines) - 1} matchups (header + C(n,2) pairs)")
print()
for line in lines[:6]:
    print(line)

Total rows: 66066 matchups (header + C(n,2) pairs)

ID,Pred
2025_1101_1102,0.702973614810117
2025_1101_1103,0.17004340235399645
2025_1101_1104,0.17049459303652417
2025_1101_1105,0.7141773829158482
2025_1101_1106,0.6142295454515001


The submission CSV contains one row for every unique pair of tournament teams
(with the lower team ID listed first in the `ID` column). Upload this file
directly to Kaggle's March Machine Learning Mania competition page.